In [1]:
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm


# acknowledgements:
# mrt/ poi csv from https://github.com/BlueSkyLT/siteselect_sg/blob/main/dataset/mrt.csv

# TODO remove all these envs

# expires wed apr 01
# ONE_MAP_API_KEY = raise NotImplementedError("Don't commit your API keys! Set them as environment variables instead.")
# LTA_DATAMALL_API_KEY = raise NotImplementedError("Don't commit your API keys! Set them as environment variables instead.")



In [2]:
resale_flat_df = pd.read_csv("data/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv")
hdb_property_df = pd.read_csv("data/HDBpropertyinformation.csv")

In [18]:
hdb_property_df.columns

Index(['blk_no', 'street', 'max_floor_lvl', 'year_completed', 'residential',
       'commercial', 'market_hawker', 'miscellaneous', 'multistorey_carpark',
       'precinct_pavilion', 'bldg_contract_town', 'total_dwelling_units',
       '1room_sold', '2room_sold', '3room_sold', '4room_sold', '5room_sold',
       'exec_sold', 'multigen_sold', 'studio_apartment_sold', '1room_rental',
       '2room_rental', '3room_rental', 'other_room_rental'],
      dtype='str')

In [17]:
hdb_property_df.head()

,blk_no,street,max_floor_lvl,year_completed,residential,commercial,market_hawker,miscellaneous,multistorey_carpark,precinct_pavilion,...,3room_sold,4room_sold,5room_sold,exec_sold,multigen_sold,studio_apartment_sold,1room_rental,2room_rental,3room_rental,other_room_rental
0,1,BEACH RD,16,1970,Y,Y,N,N,N,N,...,138,1,2,0,0,0,0,0,0,0
1,1,BEDOK STH AVE 1,14,1975,Y,N,N,Y,N,N,...,204,0,2,0,0,0,0,0,0,0
2,1,CANTONMENT RD,2,2010,N,Y,N,N,N,N,...,0,0,0,0,0,0,0,0,0,0
3,1,CHAI CHEE RD,15,1982,Y,N,N,N,N,N,...,0,10,92,0,0,0,0,0,0,0
4,1,CHANGI VILLAGE RD,4,1975,Y,Y,N,N,N,N,...,54,0,1,0,0,0,0,0,0,0


In [11]:
resale_flat_df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [20]:

# Ensure all street names in resale_flat_df are present in hdb_property_df
street_names = set(hdb_property_df["street"].unique())
resale_flat_df[~resale_flat_df["street_name"].isin(street_names)]["street_name"]



Series([], Name: street_name, dtype: str)

## Extracting Lat, Long and postal code for each flat

In [23]:
      
st = "ANG MO KIO AVE 10"
blk = "406"
url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={st} {blk}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
      
headers = {"Authorization": ONE_MAP_API_KEY}
      
response = requests.get(url, headers=headers)
      
print(response.text)

{
  "found": 1,
  "totalNumPages": 1,
  "pageNum": 1,
  "results": [
    {
      "SEARCHVAL": "406 ANG MO KIO AVENUE 10 SINGAPORE 560406",
      "BLK_NO": "406",
      "ROAD_NAME": "ANG MO KIO AVENUE 10",
      "BUILDING": "NIL",
      "ADDRESS": "406 ANG MO KIO AVENUE 10 SINGAPORE 560406",
      "POSTAL": "560406",
      "X": "30288.2346631354",
      "Y": "38229.0674628187",
      "LATITUDE": "1.36200453938712",
      "LONGITUDE": "103.853879910407"
    }
  ]
}


In [28]:
import time
from tqdm import tqdm

def get_location_info(street, block, api_key, max_retries=3):
    """Fetch postal code and coordinates from OneMap API."""
    url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={street} {block}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
    headers = {"Authorization": api_key}

    res = {"postal_code": None, "latitude": None, "longitude": None, "onemap_x": None, "onemap_y": None}
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if data.get("found", 0) > 0:
                    result = data["results"][0]
                    res = {
                        "postal_code": result.get("POSTAL"),
                        "latitude": result.get("LATITUDE"),
                        "longitude": result.get("LONGITUDE"),
                        "onemap_x": result.get("X"),
                        "onemap_y": result.get("Y")
                    }
            return res
        except Exception as e:
            if attempt == max_retries - 1:
                return res
            time.sleep(1)

def fetch_all_locations(df, street_col, block_col, api_key, requests_per_minute=300):
    """
    Fetch location info for all rows in dataframe with rate limiting.
    
    Args:
        df: DataFrame with street and block columns
        street_col: Name of the street column
        block_col: Name of the block column  
        api_key: OneMap API key
        requests_per_minute: Rate limit (default 300)
    """
    delay = 60.0 / requests_per_minute  # seconds between requests (~0.2s = 5 req/sec)
    
    postal_codes = []
    latitudes = []
    longitudes = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Fetching locations"):
        info = get_location_info(row[street_col], row[block_col], api_key)
        postal_codes.append(info["postal_code"])
        latitudes.append(info["latitude"])
        longitudes.append(info["longitude"])
        time.sleep(delay)  # Rate limiting
    
    df_result = df.copy()
    df_result["postal_code"] = postal_codes
    df_result["latitude"] = latitudes
    df_result["longitude"] = longitudes
    
    return df_result

      busstop_code             description  distance_m
2915         54319  Opp Christ The King Ch   91.924399
2914         54311                 Blk 332  158.224468
2925         54371          Blk 409 Mkt/FC  204.792047
2926         54379                 Blk 475  250.097185
2916         54321                 Blk 354  276.332772
2917         54329                 Blk 420  317.929443
2963         54601       Townsville Pr Sch  319.853557
2882         54099             Opp Blk 333  327.637898
2881         54091                 Blk 331  330.126350
2964         54609   Opp Townsville Pr Sch  333.577925
2924         54369            Opp Blk 4025  425.235359
2927         54381                 Blk 443  482.527270
2955         54531           Opp Bishan Pk  491.134908
2956         54539               Bishan Pk  494.827547


In [ ]:
# Get unique street/block combinations first to avoid duplicate API calls
# 300 requests per minute limit
unique_addresses = resale_flat_df[["street_name", "block"]].drop_duplicates().reset_index(drop=True)
print(f"Unique addresses to fetch: {len(unique_addresses)}")
print(f"Estimated time: {len(unique_addresses) / 300:.1f} minutes")

Unique addresses to fetch: 9693
Estimated time: 32.3 minutes


In [3]:
# Fetch location data (uncomment to run - will take time for large datasets)
try:
    address_locations = pd.read_csv("data/address_locations.csv")  # Load from saved file to avoid re-fetching
except FileNotFoundError:
    address_locations = fetch_all_locations(
        unique_addresses, 
        street_col="street_name", 
    block_col="block", 
    api_key=ONE_MAP_API_KEY
)


# Merge back to original dataframe
resale_flat_with_coords = resale_flat_df.merge(
    address_locations, 
    on=["street_name", "block"], 
    how="left"
)

# Save to avoid re-fetching
address_locations.to_csv("data/address_locations.csv", index=False)

# Finding nearest MRT/LRT and busstop

In [ ]:
import xml.etree.ElementTree as ET
import csv

tree = ET.parse('data/bus_stops.xml')
root = tree.getroot()

rows = []
for busstop in root.findall('busstop'):
    name = busstop.get('name', '')

    # skip entries with names starting with '-' or empty names
    if name.lower().startswith('-') or not name:
        continue
    
    details_elem = busstop.find('details')
    coords = busstop.find('coordinates')
    
    details = details_elem.text if details_elem is not None else ''
    lat = coords.find('lat').text if coords is not None and coords.find('lat') is not None else ''
    long = coords.find('long').text if coords is not None and coords.find('long') is not None else ''
    
    rows.append({
        'bus_stop_code': name,
        'description': details,
        'lat': lat,
        'long': long
    })

# Write to CSV
with open('data/bus_stops.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['bus_stop_code', 'description', 'lat', 'long'])
    writer.writeheader()
    writer.writerows(rows)

print(f"Converted {len(rows)} bus stops to data/bus_stops.csv")


hello world


# Cleaning up POI.csv


In [4]:
# Load POI data and collapse boolean type columns into a single string column
poi_df = pd.read_csv("data/poi.csv", index_col=0)

# Identify boolean type columns (between 'establishment' and 'SUBZONE_NO')
bool_cols = poi_df.columns[poi_df.columns.get_loc('establishment'):poi_df.columns.get_loc('SUBZONE_NO')]

# Create poi_types column efficiently using pd.concat to avoid fragmentation
poi_types = poi_df[bool_cols].apply(
    lambda row: ','.join([col for col in bool_cols if row[col] == True and col not in ['establishment', 'point_of_interest']]), 
    axis=1
).rename('poi_types')

# Drop bool columns and concat the new column in one operation
poi_df_clean = pd.concat([poi_df.drop(columns=bool_cols), poi_types], axis=1)

poi_df_clean.head()

,place_id,name,lat,long,rating,user_ratings_total,price_level,formatted_address,global_code,compound_code,planning_area,brand,SUBZONE_NO,SUBZONE_N,SUBZONE_C,PLN_AREA_N,PLN_AREA_C,REGION_N,REGION_C,poi_types
0,ChIJ01fgzLUe2jERxlhvImcbZ7g,Quayside Isle,1.247681,103.842072,4.3,568.0,NaN,"31 Ocean Way, Singapore 098375",6PH56RXR+3R,6RXR+3R Singapore,Southern Islands,NaN,1.0,SENTOSA,SISZ01,SOUTHERN ISLANDS,SI,CENTRAL REGION,CR,shopping_mall
1,ChIJ1S4qfY8Q2jERgb68gskzUbo,Sime Darby Centre,1.336644,103.783597,3.7,437.0,NaN,"896 Dunearn Rd, Singapore 589472",6PH58QPM+MC,8QPM+MC Singapore,Bukit Timah,NaN,2.0,SWISS CLUB,BTSZ02,BUKIT TIMAH,BT,CENTRAL REGION,CR,shopping_mall
2,ChIJ1ZAIkrwZ2jERxtZGC1JnrHM,PoMo,1.300192,103.849220,3.8,1285.0,NaN,"1 Selegie Rd, Singapore 188306",6PH58R2X+3M,8R2X+3M Singapore,Rochor,NaN,8.0,SELEGIE,RCSZ08,ROCHOR,RC,CENTRAL REGION,CR,shopping_mall
3,ChIJ1ZYJOiAZ2jER1mvQqHstQII,LR boulangerie,1.293178,103.827194,4.3,12.0,NaN,"491 River Valley Rd, #01-02 valley point shopp...",6PH57RVG+7V,7RVG+7V Singapore,Tanglin,NaN,2.0,CHATSWORTH,TNSZ02,TANGLIN,TN,CENTRAL REGION,CR,"store,food,bakery"
4,ChIJ2Y1DYBI92jERlFUKKSznJrY,Tampines Hub,1.353108,103.940361,4.6,227.0,NaN,"1 Tampines Walk, Singapore 528523",6PH59W3R+64,9W3R+64 Singapore,Tampines,NaN,3.0,TAMPINES WEST,TMSZ03,TAMPINES,TM,EAST REGION,ER,shopping_mall


In [ ]:
poi_df_clean[~poi_df_clean["poi_types"].str.startswith("establishment,point_of_interest")]["poi_types"]

0                           shopping_mall
1                           shopping_mall
2                           shopping_mall
3                       store,food,bakery
4                           shopping_mall
                      ...                
8667                                 food
8668                            food,cafe
8669                      transit_station
8670    store,food,grocery_or_supermarket
8671        food,restaurant,meal_takeaway
Name: poi_types, Length: 8672, dtype: str

# Finding nearest MRT/LRT, Busstop, POI

In [5]:
# Load all datasets
# poi_df = pd.read_csv("data/poi.csv")
poi_df_clean # already loaded and processed above
poi_cols = [x for x in bool_cols if x not in ["establishment", "point_of_interest"]] # Defined above when cleaning poi_df
bus_stops_df = pd.read_csv("data/bus_stops.csv")
mrt_stations_df = pd.read_csv("data/mrt.csv")

In [136]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in meters between two coordinates using Haversine formula."""
    R = 6371000  # Earth's radius in meters
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c

def find_nearby(lat, lon, df, lat_col="lat", lon_col="long", max_distance_m=500):
    """
    Find all entries in df within max_distance_m meters of the given coordinates.
    
    Args:
        lat: Target latitude
        lon: Target longitude
        df: DataFrame with lat/long columns
        lat_col: Name of latitude column in df
        lon_col: Name of longitude column in df
        max_distance_m: Maximum distance in meters (default 500)
    
    Returns:
        DataFrame with entries within range, including 'distance_m' column.
        If none within range, returns the single nearest entry.
    """
    df = df.copy()
    df["distance_m"] = df.apply(
        lambda row: haversine_distance(lat, lon, float(row[lat_col]), float(row[lon_col])),
        axis=1
    )
    nearby = df[df["distance_m"] <= max_distance_m].sort_values("distance_m")
    
    # If nothing within range, return just the nearest
    if nearby.empty:
        return df.nsmallest(1, "distance_m")
    
    return nearby

In [140]:
def find_all(df, mrt_stations_df, bus_stops_df, poi_df, lat_col: str = "lat", long_col: str = "long", dist_m: int = 500, poi_types: list = []):
    search_for = ["mrt", "bus", *poi_types]
    # search_for = ["mrt"]
    data = {}
    for k in search_for:
        data[f"{k}_nearest"] = []
        data[f"{k}_nearby"] = []
        
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Finding nearby locations"):
        for search in search_for:

            lat, long = float(row[lat_col]), float(row[long_col])
            if search == "mrt":
                nearby = find_nearby(lat, long, mrt_stations_df, lat_col="lat", lon_col="long", max_distance_m=dist_m)
                nearby = nearby[["name", "stop_id", "line", "distance_m"]]

            elif search == "bus":
                nearby = find_nearby(lat, long, bus_stops_df, lat_col="lat", lon_col="long", max_distance_m=dist_m)
                nearby = nearby[["busstop_code", "name", "distance_m"]]
            else:
                filtered_poi_df = poi_df[poi_df["poi_types"].str.contains(search)]
                nearby = find_nearby(lat, long, filtered_poi_df, lat_col="lat", lon_col="long", max_distance_m=dist_m)
                nearby = nearby[["name", "distance_m"]]
            data[f"{search}_nearest"].append(nearby.iloc[0].to_dict() if not nearby.empty else None)
            data[f"{search}_nearby"].append(nearby.to_dict(orient="records"))
    df_result = df.copy()
    new_cols = {}
    for k in search_for:
        new_cols[f"{k}_nearest"] = data[f"{k}_nearest"]
        new_cols[f"{k}_nearby"] = data[f"{k}_nearby"]

    df_result = pd.concat([df_result, pd.DataFrame(new_cols)], axis=1)
    return df_result

In [143]:
df_result1 = find_all(resale_flat_with_coords.head(1), mrt_stations_df, bus_stops_df, poi_df_clean, lat_col="latitude", long_col="longitude", dist_m=1000, poi_types=poi_cols)

Finding nearby locations: 100%|██████████| 1/1 [00:00<00:00,  3.86it/s]


In [144]:
df_result1

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,light_rail_station_nearest,light_rail_station_nearby,city_hall_nearest,city_hall_nearby,train_station_nearest,train_station_nearby,natural_feature_nearest,natural_feature_nearby,subpremise_nearest,subpremise_nearby
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,...,"{'name': 'Changi Airport Skytrain', 'distance_...","[{'name': 'Changi Airport Skytrain', 'distance...","{'name': 'Nee Soon Town Council Head Office', ...","[{'name': 'Nee Soon Town Council Head Office',...","{'name': 'Marina South Pier', 'distance_m': 10...","[{'name': 'Marina South Pier', 'distance_m': 1...","{'name': 'Bedok Res', 'distance_m': 8577.13322...","[{'name': 'Bedok Res', 'distance_m': 8577.1332...",{'name': 'Singapore Pools @ NTUC FairPrice (Ta...,[{'name': 'Singapore Pools @ NTUC FairPrice (T...


# Claude code optimized version    

In [8]:
# ============== OPTIMIZED VERSION (BallTree) ==============
# Uses sklearn BallTree for O(log n) spatial queries instead of O(n)
# pip install scikit-learn


# I HAVE NOT TRIED THIS

from sklearn.neighbors import BallTree
import numpy as np

EARTH_RADIUS_M = 6371000  # Earth's radius in meters

def build_ball_tree(df, lat_col="lat", lon_col="long"):
    """Build a BallTree for O(log n) spatial queries instead of O(n)."""
    coords = np.radians(df[[lat_col, lon_col]].astype(float).values)
    return BallTree(coords, metric='haversine')

def find_nearby_fast(lat, lon, df, lat_col="lat", lon_col="long", max_distance_m=500, tree=None):
    """
    Fast version of find_nearby using pre-built BallTree.
    
    Args:
        lat: Target latitude
        lon: Target longitude
        df: DataFrame with lat/long columns
        lat_col: Name of latitude column in df
        lon_col: Name of longitude column in df
        max_distance_m: Maximum distance in meters (default 500)
        tree: Pre-built BallTree (required for fast queries)
    
    Returns:
        DataFrame with entries within range, including 'distance_m' column.
        If none within range, returns the single nearest entry.
    """
    if tree is None:
        raise ValueError("tree parameter is required for find_nearby_fast")
    
    query_point = np.radians([[lat, lon]])
    radius_rad = max_distance_m / EARTH_RADIUS_M
    indices, distances = tree.query_radius(query_point, r=radius_rad, return_distance=True, sort_results=True)
    
    if len(indices[0]) == 0:
        # No results within max_distance, return the single nearest
        dist, idx = tree.query(query_point, k=1)
        nearest = df.iloc[idx[0]].copy()
        nearest["distance_m"] = dist[0] * EARTH_RADIUS_M
        return nearest
    
    nearby = df.iloc[indices[0]].copy()
    nearby["distance_m"] = distances[0] * EARTH_RADIUS_M
    return nearby

In [7]:
def find_all_fast(df, mrt_stations_df, bus_stops_df, poi_df, lat_col: str = "lat", long_col: str = "long", dist_m: int = 500, poi_types: list = []):
    """Optimized version of find_all using BallTree spatial indexing (10-100x faster)."""
    search_for = ["mrt", "bus", *poi_types]
    data = {f"{k}_nearest": [] for k in search_for}
    data.update({f"{k}_nearby": [] for k in search_for})
    
    # Pre-build BallTrees for O(log n) queries instead of O(n) per row
    print("Building spatial indexes...")
    mrt_tree = build_ball_tree(mrt_stations_df, lat_col="lat", lon_col="long")
    bus_tree = build_ball_tree(bus_stops_df, lat_col="lat", lon_col="long")
    
    # Pre-filter and build trees for each POI type
    poi_trees = {}
    poi_filtered = {}
    for poi_type in poi_types:
        filtered = poi_df[poi_df["poi_types"].str.contains(poi_type, na=False)].reset_index(drop=True)
        if len(filtered) > 0:
            poi_filtered[poi_type] = filtered
            poi_trees[poi_type] = build_ball_tree(filtered, lat_col="lat", lon_col="long")
    
    # Vectorized coordinate extraction
    lats = df[lat_col].astype(float).values
    longs = df[long_col].astype(float).values
    
    for i in tqdm(range(len(df)), desc="Finding nearby locations"):
        lat, lon = lats[i], longs[i]
        
        for search in search_for:
            if search == "mrt":
                nearby = find_nearby_fast(lat, lon, mrt_stations_df, lat_col="lat", lon_col="long", max_distance_m=dist_m, tree=mrt_tree)
                nearby = nearby[["name", "stop_id", "line", "distance_m"]]
            elif search == "bus":
                nearby = find_nearby_fast(lat, lon, bus_stops_df, lat_col="lat", lon_col="long", max_distance_m=dist_m, tree=bus_tree)
                nearby = nearby[["busstop_code", "name", "distance_m"]]
            else:
                if search in poi_trees:
                    nearby = find_nearby_fast(lat, lon, poi_filtered[search], lat_col="lat", lon_col="long", max_distance_m=dist_m, tree=poi_trees[search])
                    nearby = nearby[["name", "distance_m"]]
                else:
                    nearby = pd.DataFrame(columns=["name", "distance_m"])
            
            data[f"{search}_nearest"].append(nearby.iloc[0].to_dict() if not nearby.empty else None)
            data[f"{search}_nearby"].append(nearby.to_dict(orient="records"))
    
    df_result = df.copy()
    new_cols = {}
    for k in search_for:
        new_cols[f"{k}_nearest"] = data[f"{k}_nearest"]
        new_cols[f"{k}_nearby"] = data[f"{k}_nearby"]

    df_result = pd.concat([df_result, pd.DataFrame(new_cols)], axis=1)
    return df_result

In [9]:
# Usage (same API as find_all, just use find_all_fast instead):
df_result = find_all_fast(resale_flat_with_coords, mrt_stations_df, bus_stops_df, poi_df_clean, 
                          lat_col="latitude", long_col="longitude", dist_m=1000, poi_types=poi_cols)
                        

Building spatial indexes...


Finding nearby locations: 100%|██████████| 227932/227932 [11:33:57<00:00,  5.47it/s]      


In [10]:
df_result.to_csv("data/df_results.csv", index=False)

In [152]:
poi_cols

['store',
 'food',
 'health',
 'restaurant',
 'hospital',
 'lodging',
 'finance',
 'cafe',
 'convenience_store',
 'clothing_store',
 'atm',
 'shopping_mall',
 'grocery_or_supermarket',
 'home_goods_store',
 'school',
 'bakery',
 'beauty_salon',
 'transit_station',
 'place_of_worship',
 'pharmacy',
 'meal_takeaway',
 'furniture_store',
 'tourist_attraction',
 'secondary_school',
 'supermarket',
 'doctor',
 'shoe_store',
 'dentist',
 'jewelry_store',
 'church',
 'bank',
 'primary_school',
 'electronics_store',
 'gym',
 'spa',
 'car_repair',
 'pet_store',
 'bus_station',
 'university',
 'park',
 'general_contractor',
 'subway_station',
 'real_estate_agency',
 'florist',
 'hair_care',
 'department_store',
 'hardware_store',
 'car_dealer',
 'veterinary_care',
 'travel_agency',
 'bicycle_store',
 'book_store',
 'laundry',
 'plumber',
 'meal_delivery',
 'lawyer',
 'parking',
 'mosque',
 'physiotherapist',
 'art_gallery',
 'insurance_agency',
 'bar',
 'museum',
 'storage',
 'movie_theater',
 '